# Group Analysis
Focused analysis for a specific group of clusters.
Set `GROUP_CLUSTERS` and `GROUP_NAME` in the configuration cell below.

In [3]:
import os, sys

if '..' not in sys.path:
    sys.path.append('..')
    os.chdir('..')

import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math

import src.inference as inf
from src.el_nino import read_enso_data

## Model & Analysis Configuration

In [4]:
VALIDATION   = True
config_file  = 'checkpoint/temperature-run2/params-temperature-exp-2-c30.yaml'
save_path    = 'results/temperature-exp-2-c30'
model_suffix = None

# 'climatology' : Delta P = P(k | El Nino) - P(k)
# 'neutral'     : Delta P = P(k | El Nino) - P(k | Neutral)
COMPARISON_MODE = 'neutral'

## Group Configuration
Set the group name and the list of cluster IDs belonging to this group.

In [5]:
GROUP_NAME     = 'A1 (clusters 6, 10, 26)'                  # Label for this group
GROUP_CLUSTERS = [6, 10, 26]  # Cluster IDs in this group

## Data Loading

In [6]:
params, dataset = inf.read_data(config_file, validation=VALIDATION)
target_encoder, prot = inf.load_model(params, model_suffix=model_suffix)

E, F = inf.get_model_results(
    read_path=save_path, params=params, dataset=dataset,
    encoder=target_encoder, prototypes=prot)

if VALIDATION:
    E_val, F_val = inf.get_model_results(
        read_path=save_path, params=params, dataset=dataset.validation_imgs,
        encoder=target_encoder, prototypes=prot, validation=True)

    tsne_E, tsne_prot, tsne_Eval = inf.get_TSNE(
        save_path, E=E.cpu(), prot=prot.cpu(),
        E_val=E_val.cpu() if VALIDATION else None, validation=VALIDATION)

INFO:root:data loader created


RuntimeError: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

In [ ]:
cluster_prob, cluster_id = torch.max(F, dim=1)
df = pd.DataFrame({
    'cluster_id':   cluster_id.cpu(),
    'cluster_prob': cluster_prob.cpu(),
    'sample_type':  'train',
    'date':         dataset.time,
})

if VALIDATION:
    val_cluster_prob, val_cluster_id = torch.max(F_val, dim=1)
    df_val = pd.DataFrame({
        'cluster_id':   val_cluster_id.cpu(),
        'cluster_prob': val_cluster_prob.cpu(),
        'sample_type':  'val',
        'date':         dataset.val_time,
    })
    df = pd.concat([df, df_val]).sort_values('date').reset_index(drop=True)

    combined_dataset = torch.cat((dataset[:][0], dataset.validation_imgs))[df.index]
    mean = torch.tensor(params['data']['norm_means']).view(2, 1, 1)
    std  = torch.tensor(params['data']['norm_stds']).view(2, 1, 1)
    combined_dataset = (combined_dataset * std) + mean
else:
    combined_dataset = dataset[:][0]

nvars      = combined_dataset.shape[1]
N_PROTO    = params['criterion']['num_proto']
vars_names = params['data']['surf_vars']
var_stats  = {v: (combined_dataset[:, v].min(), combined_dataset[:, v].max()) for v in range(nvars)}

oni_index  = read_enso_data()
df['date_period'] = df['date'].dt.to_period('M')
df_el_nino = df.copy()
df_el_nino = df_el_nino.merge(oni_index, left_on='date_period', right_index=True, how='left')
df_el_nino[['Average Tmin', 'Average Tmax']] = combined_dataset.mean(dim=(2, 3))

PALETTE30 = sns.color_palette('tab20', 20) + sns.color_palette('bright', 10)

## Helper Functions

In [ ]:
def compute_anomaly(data, comparison_mode):
    prob = pd.crosstab(data['Label'], data['cluster_id'], normalize='index') * 100
    if 'El Nino' not in prob.index and 'El Niño' not in prob.index:
        raise ValueError('No El Nino rows found')
    nino_label = 'El Niño' if 'El Niño' in prob.index else 'El Nino'
    p_nino = prob.loc[nino_label]
    if comparison_mode == 'climatology':
        baseline = data['cluster_id'].value_counts(normalize=True) * 100
        baseline = baseline.reindex(p_nino.index, fill_value=0)
    else:
        neutro_label = 'Neutro' if 'Neutro' in prob.index else 'Neutral'
        baseline = prob.loc[neutro_label]
    return p_nino - baseline


_ENSO_COLS = ['Label', 'ONI', 'is_target', 'label_color']


def _build_lagged_df(df_el_nino, oni_index, lag, year_start=None):
    df = df_el_nino.copy()
    if year_start:
        df = df[df['date'].dt.year >= year_start]
    df = df.drop(columns=_ENSO_COLS, errors='ignore')
    df['month'] = df['date'].dt.month
    return df.merge(oni_index.shift(lag).dropna(),
                    left_on='date_period', right_index=True, how='left')


def _make_pivot(climate_anon, row_name, col_name='Lag'):
    anom_df = pd.DataFrame.from_dict(climate_anon, orient='index', columns=['Anomaly'])
    anom_df.index = pd.MultiIndex.from_tuples(anom_df.index, names=[row_name, col_name])
    return anom_df.reset_index().pivot(index=row_name, columns=col_name, values='Anomaly')

---
## 1. Top 3 Samples per Group Member

In [ ]:
def plot_group_top_samples(df, combined_dataset, group_clusters, group_name,
                           nvars, vars_names, top=3):
    """Show the top-N highest-probability samples for each cluster in the group."""
    n_clusters = len(group_clusters)
    n_rows = top * nvars
    gmin = combined_dataset.min().item()
    gmax = combined_dataset.max().item()

    fig, axes = plt.subplots(n_rows, n_clusters, figsize=(3 * n_clusters, 2.5 * n_rows),
                             squeeze=False)
    plt.subplots_adjust(hspace=0.4, wspace=0.1)

    for col, cluster in enumerate(group_clusters):
        top_samples = df[(df['cluster_id'] == cluster) & (df['sample_type'] == 'train')] \
            .nlargest(top, 'cluster_prob')
        for s_idx, im_index in enumerate(top_samples.index):
            image_data = combined_dataset[im_index]
            for var in range(nvars):
                ax = axes[s_idx * nvars + var, col]
                ax.imshow(image_data[var], cmap='coolwarm', vmin=gmin, vmax=gmax)
                if s_idx * nvars + var == 0:
                    ax.set_title(f'Cluster {cluster}', fontsize=10, fontweight='bold')
                if col == 0:
                    ax.set_ylabel(f'S{s_idx + 1}\n{vars_names[var]}',
                                  rotation=0, labelpad=30, va='center', fontsize=8)

    # Horizontal dividers between samples
    for r in range(nvars, n_rows, nvars):
        upper_ax = axes[r - 1, 0].get_position()
        lower_ax = axes[r,     0].get_position()
        y_pos = (upper_ax.y0 + lower_ax.y1) / 2
        fig.add_artist(plt.Line2D([0.1, 0.9], [y_pos, y_pos],
                                  transform=fig.transFigure,
                                  color='black', linewidth=1.5, linestyle='--'))

    for ax in axes.flatten():
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_visible(False)

    fig.suptitle(f'Top {top} Samples — Group {group_name} (Clusters: {group_clusters})',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()


plot_group_top_samples(df, combined_dataset, GROUP_CLUSTERS, GROUP_NAME,
                       nvars, vars_names, top=3)

---
## 2. QQ Plot — Group Members

In [ ]:
def plot_group_quantile(df, combined_dataset, group_clusters, group_name,
                        vars_names, n_quantiles=200):
    """Q-Q style deviation from the global distribution for clusters in the group."""
    q = np.linspace(0.01, 0.99, n_quantiles)
    linestyles = ['solid', 'dashed', 'dotted', 'dashdot']
    colors = sns.color_palette('bright', n_colors=len(group_clusters))

    fig, axes = plt.subplots(1, len(vars_names), figsize=(6 * len(vars_names), 5),
                             sharey=True)
    if len(vars_names) == 1:
        axes = [axes]

    for var, (ax, var_name) in enumerate(zip(axes, vars_names)):
        global_q = np.quantile(combined_dataset[:, var].ravel(), q)
        for cluster, color in zip(group_clusters, colors):
            idx  = df[df['cluster_id'] == cluster].index.values
            data = combined_dataset[:, var][idx].ravel()
            ax.plot(q, np.quantile(data, q) - global_q,
                    alpha=0.8, color=color, linestyle=linestyles[var % len(linestyles)],
                    label=f'Cluster {cluster}')
        ax.axhline(0, color='black', lw=0.8)
        ax.set_xlabel('Quantile')
        ax.set_title(f'{var_name}')
        ax.legend(loc='upper left')
        ax.set_ylabel('Delta Quantile (Cluster − Global) °C')

    fig.suptitle(f'QQ Deviation from Global Distribution — Group {group_name}',
                 fontsize=13)
    plt.tight_layout()
    plt.show()


plot_group_quantile(df, combined_dataset, GROUP_CLUSTERS, GROUP_NAME, vars_names)

---
## 3. Anomaly with Lags — Group Members

In [ ]:
def get_lagged_anomaly(df, oni_index, comparison_mode='neutral'):
    anomalias = {}
    for lag in range(-12, 13):
        merged = df.merge(oni_index.shift(lag).dropna(),
                          left_on='date_period', right_index=True, how='left')
        try:
            anomalias[lag] = compute_anomaly(merged, comparison_mode)
        except (ValueError, KeyError):
            continue
    anom = pd.DataFrame(anomalias).sort_index(axis=1)
    anom.columns.name = 'Lag'
    return anom


def plot_group_lagged_response(df, oni_index, group_clusters, group_name,
                               comparison_mode='neutral', year_filter=None):
    """Line plot of lagged ENSO anomaly for each cluster in the group."""
    cur_df = df.copy()
    if year_filter:
        cur_df = cur_df[cur_df['date'].dt.year.between(*year_filter)]

    anom = get_lagged_anomaly(cur_df, oni_index, comparison_mode)

    # Keep only clusters present in anomaly index
    available = [c for c in group_clusters if c in anom.index]

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = sns.color_palette('bright', n_colors=len(available))

    for cluster, color in zip(available, colors):
        ax.plot(anom.columns, anom.loc[cluster],
                marker='o', lw=2, alpha=0.85, color=color, label=f'Cluster {cluster}')

    ax.axhline(0, color='black', lw=1, linestyle='--', alpha=0.4)
    ax.axvline(0, color='black', lw=1, linestyle='--', alpha=0.4)
    ax.grid(True, alpha=0.4)
    ax.set_xlabel('ENSO Lag (months)')
    ax.set_ylabel('Anomaly (%)')
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)

    year_s = cur_df['date'].dt.year.min()
    year_e = cur_df['date'].dt.year.max()
    ax.set_title(f'Lagged ENSO Response — Group {group_name} ({year_s}–{year_e})',
                 fontsize=13)
    plt.tight_layout()
    plt.show()


plot_group_lagged_response(df, oni_index, GROUP_CLUSTERS, GROUP_NAME,
                           comparison_mode=COMPARISON_MODE)

---
## 4. Lagged Heatmap — All Group Members Aggregated (Cluster × Lag)

In [ ]:
def plot_group_lagged_heatmap(df_el_nino, oni_index, group_clusters, group_name,
                              year_start=None, comparison_mode='neutral'):
    """Heatmap (Cluster × Lag) restricted to clusters in the group."""
    if year_start is None:
        year_start = df_el_nino['date'].dt.year.min()
    year_end = df_el_nino['date'].dt.year.max()

    climate_anon = {}
    for lag in range(-12, 13):
        merged = _build_lagged_df(df_el_nino, oni_index, lag, year_start)
        for cur_cluster in group_clusters:
            nino_m = merged[merged['Label'] == 'El Niño']
            if comparison_mode == 'neutral':
                base_m = merged[merged['Label'] == 'Neutro']
            else:
                base_m = merged
            p_nino = (nino_m['cluster_id'] == cur_cluster).sum() / len(nino_m) * 100 if len(nino_m) > 0 else 0.0
            p_base = (base_m['cluster_id'] == cur_cluster).sum() / len(base_m) * 100 if len(base_m) > 0 else 0.0
            climate_anon[(cur_cluster, lag)] = p_nino - p_base

    pivot = _make_pivot(climate_anon, row_name='Cluster')

    fig_h = max(3, len(group_clusters) * 0.8)
    plt.figure(figsize=(14, fig_h))
    sns.heatmap(pivot, cmap='RdBu_r', center=0,
                annot=True, fmt='.2f', linewidths=.5,
                cbar_kws={'label': 'Anomaly (%)'})
    plt.title(f'Lagged Response to El Niño — Group {group_name} ({year_start}–{year_end})',
              fontsize=13)
    plt.ylabel('Cluster ID')
    plt.xlabel('ENSO Lag (months)')
    plt.tight_layout()
    plt.show()


plot_group_lagged_heatmap(df_el_nino, oni_index, GROUP_CLUSTERS, GROUP_NAME,
                          comparison_mode=COMPARISON_MODE)

---
## 5. Monthly Heatmap (Month × Lag) — per Group Member

In [ ]:
def plot_group_monthly_heatmaps(df_el_nino, oni_index, group_clusters, group_name,
                                year_start=None, comparison_mode='neutral'):
    """One Month × Lag heatmap for each cluster in the group."""
    if year_start is None:
        year_start = df_el_nino['date'].dt.year.min()
    year_end = df_el_nino['date'].dt.year.max()

    for cur_cluster in group_clusters:
        climate_anon = {}
        for lag in range(-12, 13):
            merged = _build_lagged_df(df_el_nino, oni_index, lag, year_start)
            for month in range(1, 13):
                mask_m = merged['month'] == month
                nino_m   = merged[mask_m & (merged['Label'] == 'El Niño')]
                neutral_m = merged[mask_m & (merged['Label'] == 'Neutro')]

                n_nino    = len(nino_m)
                n_neutral = len(neutral_m)

                if n_nino < 10 or n_neutral < 10:
                    climate_anon[(month, lag)] = np.nan
                    continue

                if comparison_mode == 'neutral':
                    p_nino = (nino_m['cluster_id'] == cur_cluster).sum() / n_nino * 100
                    p_base = (neutral_m['cluster_id'] == cur_cluster).sum() / n_neutral * 100
                else:
                    base_m = merged[mask_m]
                    p_nino = (nino_m['cluster_id'] == cur_cluster).sum() / n_nino * 100
                    p_base = (base_m['cluster_id'] == cur_cluster).sum() / len(base_m) * 100 if len(base_m) > 0 else 0.0

                climate_anon[(month, lag)] = p_nino - p_base

        pivot = _make_pivot(climate_anon, row_name='Month')

        plt.figure(figsize=(15, 5))
        sns.heatmap(pivot, cmap='RdBu_r', center=0,
                    annot=True, fmt='.2f', linewidths=.5,
                    cbar_kws={'label': 'Anomaly (%)'})
        plt.title(
            f'Monthly Lagged Response — Cluster {cur_cluster} '
            f'(Group {group_name}, {year_start}–{year_end})',
            fontsize=13)
        plt.ylabel('Month')
        plt.xlabel('ENSO Lag (months)')
        plt.tight_layout()
        plt.show()


plot_group_monthly_heatmaps(df_el_nino, oni_index, GROUP_CLUSTERS, GROUP_NAME,
                            comparison_mode=COMPARISON_MODE)